## Prelude

This is a recap of where we left off last lecture.

In [ ]:
import pandas as pd

df = pd.read_csv("https://dlsun.github.io/pods/data/bordeaux.csv",
                 index_col="year")
df

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

fig1 = px.scatter(df[~df["price"].isnull()],
                  x="win", y="summer", color="price")
fig2 = px.scatter(df[df["price"].isnull()],
                  x="win", y="summer", symbol_sequence=["circle-open"])

go.Figure(data=fig1.data + fig2.data, layout=fig1.layout)

## $K$-Nearest Neighbors from Scratch

In [ ]:
df_train = df.loc[:1980].copy()
df_test = df.loc[1981:].copy()

Scale the input variables.

In [ ]:
X_train = df_train[["win", "summer"]]
y_train = df_train["price"]

# Standardize the features.
X_train_mean = X_train.mean()
X_train_sd = X_train.std()
X_train_scaled = (X_train - X_train_mean) / X_train_sd

In [ ]:
X_test = df_test[["win", "summer"]]
X_test_scaled = (X_test - X_train_mean) / X_train_sd
X_test_scaled

Calculate distances and find the 5 closest training observations.

In [ ]:
import numpy as np
dists = np.sqrt(
    ((X_test_scaled.loc[1986] - X_train_scaled) ** 2).sum(axis=1))
dists

In [ ]:
index_nearest = dists.sort_values().index[:5]
index_nearest

Finally, to make a prediction, we average the labels $y$ of these "nearest" training observations.

In [ ]:
y_train[index_nearest].mean()

We can do this for every vintage in the test data by writing a `for` loop.

In [ ]:
def calculate_knn_prediction(test_obs_scaled):
  # TODO: determine the k-nearest neighbors to test_obs_scaled
  # TODO: calculate the mean label of the k-nearest neighbors
  pass

for year in range(1981, 1992):
  print(year, calculate_knn_prediction(X_test_scaled.loc[year]))

If we want the predictions in a Pandas object, we can use the `.apply()` function.

In [ ]:
X_test_scaled.apply(calculate_knn_prediction, axis="columns")

# $K$-Nearest Neighbors in Scikit-Learn

Scikit-learn provides a built-in model `KNeighborsRegressor` that fits $k$-nearest neighbors regression models.

But first, we need to scale the training and test data.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)

# Scale the test data using the scaler that was fit to the training data!
X_test_scaled = scaler.transform(X_test)

In [ ]:
from sklearn.neighbors import KNeighborsRegressor

model = KNeighborsRegressor(n_neighbors=5)
model.fit(X=X_train_scaled, y=y_train)
model.predict(X=X_test_scaled)

A `Pipeline` allows us to chain together preprocessing and modeling steps. They have `.fit()` and `.predict()` methods like any model.

In [ ]:
from sklearn.pipeline import make_pipeline

pipeline = make_pipeline(
          StandardScaler(),
          KNeighborsRegressor(n_neighbors=5))
pipeline.fit(X=X_train, y=y_train)
pipeline.predict(X=X_test)